# [6.4] Crosscoders and Model Diffing - Solutions

This notebook runs the solved crosscoder/model-diffing contracts, then displays the report-backed `gelu-1l` vs `solu-1l` signature result. Keep the claim boundary in view: this is a scoped paired-checkpoint preflight, not a trained crosscoder replication.

<details>
<summary>Expected output</summary>

The local tests should all print pass messages. The final table and plots should match the committed `verification_report.json` metrics: exact reconstruction, top variance fraction near `0.805`, technical/everyday label AUC `1.0`, and direction-removal reduction much larger than the orthogonal random control.

</details>

<details>
<summary>Help - why these controls matter</summary>

Model-diffing explanations are easy to overread. Reconstruction, signed specificity, label prediction, paired deltas, and direction-removal controls remove different failure modes before the real-model result is interpreted.

</details>


In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter6_sparse_feature_methods"
section = "part4_crosscoders_model_diffing"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_crosscoders_model_diffing.tests as tests
import part4_crosscoders_model_diffing.solutions as solutions

CrosscoderOutput = solutions.CrosscoderOutput
CrosscoderReconstructionReport = solutions.CrosscoderReconstructionReport
FeatureSpecificityReport = solutions.FeatureSpecificityReport
BehaviorDeltaPredictionReport = solutions.BehaviorDeltaPredictionReport
CrosscoderAblationReport = solutions.CrosscoderAblationReport
decode_crosscoder = solutions.decode_crosscoder
crosscoder_reconstruction_report = solutions.crosscoder_reconstruction_report
feature_specificity_report = solutions.feature_specificity_report
classify_features_by_specificity = solutions.classify_features_by_specificity
roc_auc_binary = solutions.roc_auc_binary
behavior_delta_prediction_report = solutions.behavior_delta_prediction_report
toy_behavior_delta_scores = solutions.toy_behavior_delta_scores
crosscoder_ablation_report = solutions.crosscoder_ablation_report
run_smoke_test = solutions.run_smoke_test


## Local Tests

These tests cover exact shared/model-specific reconstruction, signed feature ownership, signed AUC prediction, model-B-minus-model-A deltas, and target-vs-random controls.


In [ ]:
tests.test_decode_crosscoder_reconstructs_shared_and_specific_spaces(
    decode_crosscoder,
    crosscoder_reconstruction_report,
)
tests.test_feature_specificity_classifies_shared_model_a_and_model_b(
    feature_specificity_report,
    classify_features_by_specificity,
)
tests.test_behavior_delta_prediction_uses_signed_auc_and_means(
    behavior_delta_prediction_report,
    roc_auc_binary,
)
tests.test_toy_behavior_delta_scores_are_model_b_minus_model_a(
    toy_behavior_delta_scores,
)
tests.test_crosscoder_ablation_requires_target_to_beat_random_control(
    crosscoder_ablation_report,
)


## Whole-Notebook Contract

The smoke-test contract combines the local components into the dictionary shape used by the verification report.


In [ ]:
tests.test_notebook_contract(run_smoke_test)
contract = run_smoke_test(cpu=True)
contract


## Signature Result

<details>
<summary>Interpreting the signature result</summary>

The report validates a pinned TransformerLens paired-checkpoint path. The top SVD direction over residual deltas separates the generated technical/everyday prompt labels, and removing that direction reduces activation-delta norm much more than removing an orthogonal random direction. This is activation-space model diffing evidence, not a broad causal behavior claim.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


def signature_table(gpu: dict) -> list[tuple[str, object]]:
    return [
        ("model A / B", f"{gpu['model_a_name']} / {gpu['model_b_name']}"),
        ("model A revision", gpu["model_a_revision"][:12]),
        ("model B revision", gpu["model_b_revision"][:12]),
        ("prompts", f"{gpu['technical_prompt_count']} technical / {gpu['everyday_prompt_count']} everyday"),
        ("activation shape", gpu["activation_shape"]),
        ("model A / B MSE", f"{gpu['model_a_mse']:.2e} / {gpu['model_b_mse']:.2e}"),
        ("shared active fraction", round(gpu["shared_active_fraction"], 3)),
        ("top singular value", round(gpu["top_singular_value"], 3)),
        ("top variance fraction", round(gpu["top_variance_fraction"], 3)),
        ("technical/everyday label AUC", round(gpu["behavior_delta_auc"], 3)),
        ("top-direction projection means", f"{gpu['behavior_delta_positive_mean']:.3f} / {gpu['behavior_delta_negative_mean']:.3f}"),
        ("baseline delta norm", round(gpu["baseline_delta_norm"], 3)),
        ("top-direction removed norm", round(gpu["top_direction_ablated_delta_norm"], 3)),
        ("orthogonal-control removed norm", round(gpu["random_direction_ablated_delta_norm"], 3)),
        ("activation-delta reduction", round(gpu["delta_reduction"], 3)),
        ("random-direction reduction", round(gpu["random_reduction"], 3)),
        ("floor-vs-top diagnostic abs mean", round(gpu["floor_top_delta_abs_mean"], 3)),
        ("peak VRAM GB", round(gpu["peak_vram_gb"], 3)),
    ]


gpu = run_gpu_test(max_vram_gb=24.0)
signature_table(gpu)


In [ ]:
import matplotlib.pyplot as plt

gpu = run_gpu_test(max_vram_gb=24.0)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

axes[0].bar(
    ["model A", "model B"],
    [gpu["model_a_mse"], gpu["model_b_mse"]],
    color=["#2563eb", "#0f766e"],
)
axes[0].set_title("Exact reconstruction")
axes[0].set_ylabel("MSE, lower is better")

axes[1].bar(
    ["technical", "everyday"],
    [gpu["behavior_delta_positive_mean"], gpu["behavior_delta_negative_mean"]],
    color=["#16a34a", "#94a3b8"],
)
axes[1].set_title(f"Top direction label AUC = {gpu['behavior_delta_auc']:.3f}")
axes[1].set_ylabel("projection mean")

axes[2].bar(
    ["baseline", "top removed", "orthogonal"],
    [
        gpu["baseline_delta_norm"],
        gpu["top_direction_ablated_delta_norm"],
        gpu["random_direction_ablated_delta_norm"],
    ],
    color=["#2563eb", "#16a34a", "#f97316"],
)
axes[2].set_title("Direction-removal control")
axes[2].set_ylabel("activation-delta norm")
axes[2].tick_params(axis="x", rotation=15)

fig.tight_layout()
plt.show()


## Limitations

The report proves a scoped local preflight on two small public checkpoints and 16 generated safe prompts. It uses an exact shared-plus-delta construction and SVD, not a trained sparse crosscoder. It does not establish learned model-specific features, broad prompt robustness, generated-completion changes, or causal behavior control.
